In [1]:
import sys
!{sys.executable} -m pip install google-generativeai pandas

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https:

In [1]:
api_key="Your_API_key"

In [20]:
import pandas as pd
df = pd.read_csv("HR-Employee-Attrition.xlsx.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,3,4,1,6,3,3,2,2,2,2


In [4]:
total_employees = len(df)
attrition_rate = (df['Attrition'] == 'Yes').mean() * 100
dept_attrition = df.groupby('Department')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100)

print("Total Employees:", total_employees)
print("Overall Attrition Rate:", round(attrition_rate, 2), "%")
print("Attrition by Department:")
print(dept_attrition)

Total Employees: 1470
Overall Attrition Rate: 16.12 %
Attrition by Department:
Department
Human Resources           19.047619
Research & Development    13.839750
Sales                     20.627803
Name: Attrition, dtype: float64


In [21]:
import google.generativeai as genai

genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-flash-latest")

def ask_question(question):
    context = f"""
    Total Employees: {total_employees}
    Overall Attrition Rate: {round(attrition_rate, 2)}%
    Attrition by Department: {dept_attrition.to_dict()}
    """
    
    prompt = f"""You are a helpful HR data analyst. Answer the question using only the data provided below.

Data:
{context}

Question: {question}
"""
    
    response = model.generate_content(prompt)
    return response.text

In [22]:
print(ask_question("Which department has the highest attrition rate?"))

Based on the provided data, the **Sales** department has the highest attrition rate at approximately **20.63%** (20.62780269058296%).


In [10]:
print(ask_question("What percentage of employees are at high risk of leaving?"))

Based on the data provided, there is no information or metric indicating the percentage of employees who are at "high risk" of leaving. The data only details the total number of employees (1,470), the overall attrition rate (16.12%), and attrition rates by department.


In [11]:
print(ask_question("Give me one recommendation to reduce attrition in the Sales department."))

Based strictly on the data provided, it is not possible to give a specific recommendation to reduce attrition in the Sales department. The data only shows headcount and attrition rates (highlighting that Sales has the highest attrition rate at 20.63%), but it does not include any information regarding the causes, drivers, or context behind why employees are leaving (such as compensation, job satisfaction, workload, or tenure). 

To formulate an effective recommendation, additional data on the reasons for attrition in Sales would be required.


In [16]:
print(df.columns.tolist())

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [17]:
def ask_question(question):
    total_employees = df.shape[0]
    attrition_rate = (df['Attrition'] == 'Yes').mean() * 100
    dept_attrition = df[df['Attrition'] == 'Yes']['Department'].value_counts()
    overtime_attrition = df[df['Attrition'] == 'Yes']['OverTime'].value_counts()
    role_attrition = df[df['Attrition'] == 'Yes']['JobRole'].value_counts()

    context = f"""
    Total Employees: {total_employees}
    Overall Attrition Rate: {round(attrition_rate, 2)}%
    Attrition Count by Department: {dept_attrition.to_dict()}
    Attrition Count by OverTime Status: {overtime_attrition.to_dict()}
    Attrition Count by Job Role: {role_attrition.to_dict()}
    """

    prompt = f"""You are a helpful HR data analyst. Answer the question using only the data provided below. If the answer isn't in the data, say you don't have that information.

Data:
{context}

Question: {question}
"""

    response = model.generate_content(prompt)
    return response.text

In [18]:
print(ask_question("Which job role has the highest attrition?"))

Based on the provided data, the **Laboratory Technician** job role has the highest attrition, with an attrition count of 62.
